# WB_HCP Data360 → landing_dev (Stage 1)

Discover and review here. After the notebook and `landing_dev.wb_hcp_data360` look good, promote via the package into **`landing`** (Stage 2).

- Guide: [`wb_hcp_data360.md`](./wb_hcp_data360.md)
- Preferred env: `data-eng/venv` (has deps). First cell also bootstraps `ingestion` if the kernel is wrong.
- Creds: `data-eng/ingestion/.env` (project, location `europe-west3`, SA key outside the repo)

## Setup (landing_dev)

Bootstraps `ingestion` onto `sys.path` (no special kernel required), then loads settings.

In [1]:
import sys
from pathlib import Path


def _load_notebook_bootstrap() -> None:
    """Find notebooks/_bootstrap.py from cwd or workspace and ensure_ingestion()."""
    here = Path.cwd().resolve()
    candidates: list[Path] = []
    for base in [here, *here.parents]:
        candidates.extend(
            [
                base / "_bootstrap.py",
                base / "notebooks" / "_bootstrap.py",
                base / "ingestion" / "notebooks" / "_bootstrap.py",
                base / "data-eng" / "ingestion" / "notebooks" / "_bootstrap.py",
            ]
        )
    for boot in candidates:
        if boot.is_file():
            nb_dir = str(boot.parent)
            if nb_dir not in sys.path:
                sys.path.insert(0, nb_dir)
            from _bootstrap import ensure_ingestion

            pkg = ensure_ingestion()
            print("ingestion ready", pkg)
            return
    raise ModuleNotFoundError(
        "Could not find notebooks/_bootstrap.py. "
        "Open this notebook from the data-team / data-eng repo tree."
    )


_load_notebook_bootstrap()

from ingestion.common.auth import ensure_auth_safe
from ingestion.common.bq import get_client
from ingestion.common.config import get_settings
from ingestion.common.load import add_metadata, load_dataframe, new_run_id

TABLE = "wb_hcp_data360"
SOURCE_NAME = "wb_hcp_data360"
SOURCE_URL = "https://data360api.worldbank.org/data360/data"
KEY_COLS = ["database_id", "indicator", "ref_area", "sex", "age", "time_period"]
START_YEAR = 2020

settings = get_settings(cli_target="landing_dev")
ensure_auth_safe(settings)
run_id = new_run_id()
print("python", sys.executable)
print("target", settings.target_dataset, "project", settings.project, "run_id", run_id)

ingestion ready C:\Users\chidi\projects\openTrace\data-team\data-eng\ingestion\package
python c:\Users\chidi\projects\openTrace\data-team\ml-eng\venv\Scripts\python.exe
target landing_dev project opentrace-prod-5ga4 run_id 93ece877-3db3-4cbb-88d6-ae71518c10b2


## Fetch Data360 (pilot slice)

Uppercase params: `DATABASE_ID`, `INDICATOR`, `REF_AREA`.

In [2]:
import requests
import pandas as pd

params = {
    "DATABASE_ID": "WB_HCP",
    "INDICATOR": "WB_HCP_EMP_2WAP_A",
    "REF_AREA": "KEN,NGA,GHA",
}
resp = requests.get(SOURCE_URL, params=params, timeout=60)
resp.raise_for_status()
payload = resp.json()
raw = pd.DataFrame(payload.get("value") or [])
print("count", payload.get("count"), "rows", len(raw))
print("areas", sorted(raw["REF_AREA"].unique()) if len(raw) else [])
raw.head()

count 210 rows 210
areas ['GHA', 'KEN', 'NGA']


,OBS_VALUE,TIME_FORMAT,UNIT_MULT,COMMENT_OBS,OBS_STATUS,OBS_CONF,AGG_METHOD,DECIMALS,COMMENT_TS,DATA_SOURCE,...,SEX,AGE,URBANISATION,COMP_BREAKDOWN_1,COMP_BREAKDOWN_2,COMP_BREAKDOWN_3,TIME_PERIOD,FREQ,UNIT_MEASURE,UNIT_TYPE
0,80.983,602,0,None,A,PU,_Z,2,None,None,...,F,_T,_Z,_Z,_Z,_Z,1990,A,PT,None
1,87.294,602,0,None,A,PU,_Z,2,None,None,...,_T,_T,_Z,_Z,_Z,_Z,1990,A,PT,None
2,81.13,602,0,None,A,PU,_Z,2,None,None,...,F,_T,_Z,_Z,_Z,_Z,1991,A,PT,None
3,87.314,602,0,None,A,PU,_Z,2,None,None,...,_T,_T,_Z,_Z,_Z,_Z,1991,A,PT,None
4,81.288,602,0,None,A,PU,_Z,2,None,None,...,F,_T,_Z,_Z,_Z,_Z,1992,A,PT,None


## Normalize (inline — same mapping as package transform)

In [3]:
COL_MAP = {
    "DATABASE_ID": "database_id",
    "INDICATOR": "indicator",
    "REF_AREA": "ref_area",
    "SEX": "sex",
    "AGE": "age",
    "URBANISATION": "urbanisation",
    "TIME_PERIOD": "time_period",
    "FREQ": "freq",
    "OBS_VALUE": "obs_value",
    "UNIT_MEASURE": "unit_measure",
    "OBS_STATUS": "obs_status",
    "DATA_SOURCE": "data_source",
}

rename = {c: COL_MAP[c] for c in raw.columns if c in COL_MAP}
df = pd.DataFrame(raw.rename(columns=rename))
keep = [c for c in COL_MAP.values() if c in df.columns]
df = pd.DataFrame(df.loc[:, keep])
time_period = pd.Series(df["time_period"], index=df.index).astype(str).str.strip()
time_period_date = pd.Series(
    pd.to_datetime(time_period + "-01-01", errors="coerce"),
    index=df.index,
)
df["obs_value"] = pd.to_numeric(pd.Series(df["obs_value"], index=df.index), errors="coerce")
df["time_period"] = time_period
df["time_period_date"] = time_period_date
df = pd.DataFrame(df.loc[time_period_date.dt.year >= START_YEAR].reset_index(drop=True))
print("rows after filter", len(df))
df.head()

rows after filter 30


,database_id,indicator,ref_area,sex,age,urbanisation,time_period,freq,obs_value,unit_measure,obs_status,data_source,time_period_date
0,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,F,_T,_Z,2020,A,76.512,PT,A,None,2020-01-01
1,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,_T,_T,_Z,2020,A,82.640,PT,A,None,2020-01-01
2,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,F,_T,_Z,2021,A,77.615,PT,A,None,2021-01-01
3,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,_T,_T,_Z,2021,A,83.511,PT,A,None,2021-01-01
4,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,F,_T,_Z,2022,A,77.510,PT,A,None,2022-01-01


## Preview / optional local parquet

In [4]:
ready = add_metadata(
    pd.DataFrame(df),
    source=SOURCE_NAME,
    run_id=run_id,
    source_url=SOURCE_URL,
    key_cols=KEY_COLS,
)
settings.out_dir.mkdir(parents=True, exist_ok=True)
preview_path = settings.out_dir / f"{SOURCE_NAME}_{run_id}.parquet"
ready.to_parquet(preview_path, index=False)
print("preview", preview_path, "cols", list(ready.columns))
ready.head()

preview C:\Users\chidi\projects\openTrace\data-team\data-eng\ingestion\.out\wb_hcp_data360_93ece877-3db3-4cbb-88d6-ae71518c10b2.parquet cols ['database_id', 'indicator', 'ref_area', 'sex', 'age', 'urbanisation', 'time_period', 'freq', 'obs_value', 'unit_measure', 'obs_status', 'data_source', 'time_period_date', '_source', '_run_id', '_ingested_at', '_ingest_date', '_source_url', '_record_hash']


,database_id,indicator,ref_area,sex,age,urbanisation,time_period,freq,obs_value,unit_measure,obs_status,data_source,time_period_date,_source,_run_id,_ingested_at,_ingest_date,_source_url,_record_hash
0,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,F,_T,_Z,2020,A,76.512,PT,A,None,2020-01-01,wb_hcp_data360,93ece877-3db3-4cbb-88d6-ae71518c10b2,2026-10-23 22:21:55.206315+00:00,2026-10-23,https://data360api.worldbank.org/data360/data,32809433b5b70b8818f0f6651ccf3a91d16c61db65e7b8...
1,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,_T,_T,_Z,2020,A,82.640,PT,A,None,2020-01-01,wb_hcp_data360,93ece877-3db3-4cbb-88d6-ae71518c10b2,2026-10-23 22:21:55.206315+00:00,2026-10-23,https://data360api.worldbank.org/data360/data,3a7223d2d23bc579ca184ead7ae1364839d48a2ea002e5...
2,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,F,_T,_Z,2021,A,77.615,PT,A,None,2021-01-01,wb_hcp_data360,93ece877-3db3-4cbb-88d6-ae71518c10b2,2026-10-23 22:21:55.206315+00:00,2026-10-23,https://data360api.worldbank.org/data360/data,f46fe08c47d7ed89f1e019e6a63833d34aa165699b06b8...
3,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,_T,_T,_Z,2021,A,83.511,PT,A,None,2021-01-01,wb_hcp_data360,93ece877-3db3-4cbb-88d6-ae71518c10b2,2026-10-23 22:21:55.206315+00:00,2026-10-23,https://data360api.worldbank.org/data360/data,46e5178d2da6e60b5abd54bb4991aa567877a1dcd251c4...
4,WB_HCP,WB_HCP_EMP_2WAP_A,KEN,F,_T,_Z,2022,A,77.510,PT,A,None,2022-01-01,wb_hcp_data360,93ece877-3db3-4cbb-88d6-ae71518c10b2,2026-10-23 22:21:55.206315+00:00,2026-10-23,https://data360api.worldbank.org/data360/data,4d158b4ab60371c3a67045c9cbfe45763d6d90d5470114...


## Load into landing_dev

Uses the shared load job (`replace_partition`), not `insert_rows_json`.

In [5]:
client = get_client(settings)
result = load_dataframe(
    client,
    settings,
    ready,
    table=TABLE,
    mode="replace_partition",
    run_id=run_id,
)
print(result)

c:\Users\chidi\projects\openTrace\data-team\ml-eng\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadResult(rows=30, mode='replace_partition', destination='opentrace-prod-5ga4.landing_dev.wb_hcp_data360$20261023', staging=None, merge_sql=None, dropped_staging=False)


## After review → Stage 2 (package → landing)

Not this notebook’s job. Once the notebook and `landing_dev.wb_hcp_data360` are reviewed:

- Package: `ingestion/package/ingestion/sources/wb_hcp_data360/`
- CI/prod: `python -m ingestion run wb_hcp_data360` with target `landing`
  (requires `INGEST_ALLOW_PROD=1` and `CI=true`)

Local package dry-run against fixtures (optional):  
`INGEST_OFFLINE_FIXTURES=1 python -m ingestion run wb_hcp_data360 --dry-run`